In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import kagglehub
import clip
from transformers import CvtModel
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import gradio as gr
import matplotlib.pyplot as plt

In [ ]:
FINAL_MODEL_PATH = # Path 
TEMP_FINAL_PATH = # Path

In [ ]:
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
class CIFAKE(Dataset):
  def __init__(self, root_dir, transform=None):
    self.image_paths = []
    self.labels = []
    self.transform = transform

    for label_name in ["REAL", "FAKE"]:
      label = 0 if label_name == "REAL" else 1
      folder = os.path.join(root_dir, label_name)

      if not os.path.exists(folder):
        continue

      for image_name in os.listdir(folder):
        self.image_paths.append(os.path.join(folder, image_name))
        self.labels.append(label)

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, index):
    img = Image.open(self.image_paths[index]).convert("RGB")
    label = torch.tensor(self.labels[index], dtype=torch.float32)

    if self.transform:
      img = self.transform(img)

    return img, label

In [ ]:
CLIP_MEAN = [
    0.48145466,
    0.4578275,
    0.40821073
]

CLIP_STD = [
    0.26862954,
    0.26130258,
    0.27577711
]

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=CLIP_MEAN,
        std=CLIP_STD
    )
])

train_path = os.path.join(path, "train")
test_path = os.path.join(path, "test")

train_dataset = CIFAKE(train_path, transform=transform)
test_dataset = CIFAKE(test_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
# CLIP
clip_model, _ = clip.load("ViT-B/32", device=device)
for p in clip_model.parameters():
  p.requires_grad = False

# CvT
cvt_model = CvtModel.from_pretrained("microsoft/cvt-13").to(device)

In [ ]:
class FusionDetector(nn.Module):
    def __init__(self, cvt_model, clip_model):
        super(FusionDetector, self).__init__()
        self.cvt = cvt_model
        self.clip = clip_model

        dummy_input = torch.randn(1, 3, 224, 224).to(device)

        with torch.no_grad():
            cvt_outputs = self.cvt(pixel_values=dummy_input)
            self.cvt_feature_dim = cvt_outputs.last_hidden_state.mean(dim=1).flatten(start_dim=1).view(1, -1).shape[1]

            self.clip_feature_dim = self.clip.encode_image(dummy_input).flatten(start_dim=1).view(1, -1).shape[1]

        print(f"Detected CvT Feature Dimension: {self.cvt_feature_dim}")
        print(f"Detected CLIP Feature Dimension: {self.clip_feature_dim}")

        self.classifier = nn.Sequential(
            nn.Linear(self.cvt_feature_dim + self.clip_feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        cvt_outputs = self.cvt(pixel_values=x)
        cvt_features = cvt_outputs.last_hidden_state.mean(dim=1)
        cvt_features = cvt_features.flatten(start_dim=1).view(x.size(0), -1)

        with torch.no_grad():
            clip_features = self.clip.encode_image(x)
            clip_features = clip_features / clip_features.norm(dim=-1, keepdim=True)
            clip_features = clip_features.flatten(start_dim=1).view(x.size(0), -1)

        fused_features = torch.cat((cvt_features, clip_features.float()), dim=1)

        logits = self.classifier(fused_features)
        return logits.squeeze(-1)

model = FusionDetector(cvt_model, clip_model).to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-2
)

In [ ]:
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    print(f"\n--- Epoch {epoch+1}/{epochs} ---")
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 100 == 0:
            print(f"Batch {batch_idx + 1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_acc = (correct / total) * 100
    print(f"Epoch {epoch+1} Train Loss: {running_loss/len(train_loader):.4f} | Train Accuracy: {train_acc:.2f}%")

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = (val_correct / val_total) * 100
    print(f"Validation Accuracy: {val_acc:.2f}%")

print("\nTraining completed successfully!")

In [ ]:
def evaluate_and_report(model, data_loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    print("Gathering model predictions across the test set...")
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)

            logits = model(images)

            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print("\n================ EVALUATION METRICS ================")
    acc = accuracy_score(all_labels, all_preds)
    print(f"Overall Model Accuracy: {acc * 100:.2f}%\n")

    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=["REAL", "FAKE"], digits=4))

    print("Confusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(f"True Real:  {cm[0][0]:5d}  |  False Fake: {cm[0][1]:5d}")
    print(f"False Real: {cm[1][0]:5d}  |  True Fake:  {cm[1][1]:5d}")
    print("====================================================")

evaluate_and_report(model, test_loader, device)

In [ ]:
def save_to_both_locations(model, primary_path, backup_path, label_name="final model"):
    """
    Saves the model state dictionary to both a permanent and temporary path.
    """
    os.makedirs(os.path.dirname(primary_path), exist_ok=True)

    state_dict = model.state_dict()

    try:
        torch.save(state_dict, primary_path)
        print(f" Successfully saved [{label_name}] to permanent path: {primary_path}")
    except Exception as e:
        print(f"Failed saving to primary path: {e}")

    try:
        torch.save(state_dict, backup_path)
        print(f" Successfully saved [{label_name}] to temporary path: {backup_path}")
    except Exception as e:
        print(f"Failed saving to backup path: {e}")

In [ ]:
save_to_both_locations(model, FINAL_MODEL_PATH, TEMP_FINAL_PATH, "final model")

In [ ]:

test_cvt = CvtModel.from_pretrained("microsoft/cvt-13").to(device)
test_clip, _ = clip.load("ViT-B/32", device=device)
loaded_model = FusionDetector(test_cvt, test_clip).to(device)

print(f"Loading weights from: {FINAL_MODEL_PATH}")
loaded_model.load_state_dict(torch.load(FINAL_MODEL_PATH, map_location=device))
loaded_model.eval()


In [ ]:
import random

def verify_cifake_predictions(
    model,
    dataset,
    device,
    samples=20
):

    model.eval()

    print("\nVERIFYING CIFAKE TEST IMAGES")
    print("=" * 80)

    correct = 0

    with torch.no_grad():

        for i in range(samples):

            idx = random.randint(
                0,
                len(dataset) - 1
            )

            image, label = dataset[idx]

            image = image.unsqueeze(0).to(device)

            output = model(image)

            logit = output.item()

            prob_fake = torch.sigmoid(
                output
            ).item()

            prediction = (
                1 if prob_fake >= 0.5
                else 0
            )

            if prediction == label.item():
                correct += 1

            truth = (
                "FAKE"
                if label.item() == 1
                else "REAL"
            )

            pred = (
                "FAKE"
                if prediction == 1
                else "REAL"
            )

            print(
                f"Sample {i+1:02d} | "
                f"Truth={truth:4s} | "
                f"Pred={pred:4s} | "
                f"Logit={logit:8.4f} | "
                f"FakeProb={prob_fake:.6f}"
            )

    print("=" * 80)
    print(
        f"Accuracy on sampled images: "
        f"{correct}/{samples}"
    )

In [ ]:
import matplotlib.pyplot as plt

def plot_distribution(
    model,
    loader,
    device
):

    model.eval()

    real_scores = []
    fake_scores = []

    print(
        "Collecting prediction scores..."
    )

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            logits = model(images)

            probs = torch.sigmoid(
                logits
            )

            for prob, label in zip(
                probs.cpu(),
                labels
            ):

                if label.item() == 0:

                    real_scores.append(
                        prob.item()
                    )

                else:

                    fake_scores.append(
                        prob.item()
                    )

    print(
        f"REAL samples: {len(real_scores)}"
    )

    print(
        f"FAKE samples: {len(fake_scores)}"
    )

    plt.figure(
        figsize=(10,5)
    )

    plt.hist(
        real_scores,
        bins=50,
        alpha=0.5,
        label="REAL"
    )

    plt.hist(
        fake_scores,
        bins=50,
        alpha=0.5,
        label="FAKE"
    )

    plt.xlabel(
        "Predicted Fake Probability"
    )

    plt.ylabel(
        "Count"
    )

    plt.title(
        "Prediction Score Distribution"
    )

    plt.legend()

    plt.show()

    print(
        f"Average REAL score: "
        f"{sum(real_scores)/len(real_scores):.4f}"
    )

    print(
        f"Average FAKE score: "
        f"{sum(fake_scores)/len(fake_scores):.4f}"
    )

In [ ]:
import os
from PIL import Image

def evaluate_folder(
    folder_path,
    model,
    device
):

    model.eval()

    print(
        f"\nEvaluating Folder:"
    )

    print(folder_path)

    print("=" * 80)

    test_transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[
                0.48145466,
                0.4578275,
                0.4821071
            ],
            std=[
                0.26862954,
                0.26130258,
                0.27577711
            ]
        )
    ])

    files = sorted(
        os.listdir(folder_path)
    )

    with torch.no_grad():

        for filename in files:

            filepath = os.path.join(
                folder_path,
                filename
            )

            try:

                image = Image.open(
                    filepath
                ).convert("RGB")

                tensor = test_transform(
                    image
                ).unsqueeze(0).to(device)

                output = model(
                    tensor
                )

                logit = output.item()

                prob_fake = torch.sigmoid(
                    output
                ).item()

                prediction = (
                    "FAKE"
                    if prob_fake >= 0.5
                    else "REAL"
                )

                print(
                    f"{filename:30s}"
                    f" | {prediction:4s}"
                    f" | Logit={logit:8.4f}"
                    f" | FakeProb={prob_fake:.6f}"
                )

            except Exception as e:

                print(
                    f"{filename}: {e}"
                )

In [ ]:
verify_cifake_predictions(
    loaded_model,
    test_dataset,
    device
)

plot_distribution(
    loaded_model,
    test_loader,
    device
)

evaluate_folder(
    #"path",
    loaded_model,
    device
)

evaluate_folder(
    #"path",
    loaded_model,
    device
)

In [ ]:
def test_single_image(
    image_file_path,
    model_instance,
    evaluation_device
):

    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.48145466, 0.4578275, 0.4821071],
            std=[0.26862954, 0.26130258, 0.27577711]
        )
    ])

    try:

        img = Image.open(
            image_file_path
        ).convert("RGB")

        tensor = test_transform(
            img
        ).unsqueeze(0).to(
            evaluation_device
        )

        model_instance.eval()

        with torch.no_grad():

            output_logit = model_instance(
                tensor
            )

            logit = output_logit.item()

            probability = torch.sigmoid(
                output_logit
            ).item()

            real_probability = (
                1 - probability
            )

        print("\n========================")
        print(
            f"Target File: "
            f"{os.path.basename(image_file_path)}"
        )

        print(
            f"Raw Logit: "
            f"{logit:.4f}"
        )

        print(
            f"Fake Probability: "
            f"{probability:.6f}"
        )

        print(
            f"Real Probability: "
            f"{real_probability:.6f}"
        )

        if probability >= 0.5:

            print(
                f"Prediction: AI-Generated"
            )

        else:

            print(
                f"Prediction: Real"
            )

        print("========================")

    except Exception as err:

        print(
            f"Could not check target image file path: "
            f"{err}"
        )
test_single_image(
    #"path",
    loaded_model,
    device
)

In [ ]:
def predict_uploaded_image(input_image):

    if input_image is None:
        return "Please upload an image."

    img = input_image.convert("RGB")

    current_device = next(
        loaded_model.parameters()
    ).device

    img_tensor = transform(
        img
    ).unsqueeze(0).to(
        current_device
    )

    loaded_model.eval()

    with torch.no_grad():

        logit = loaded_model(
            img_tensor
        )

        logit = logit.view(-1)

        raw_logit = logit.item()

        temperature = 3.0

        prob_fake = torch.sigmoid(
            logit / temperature
        ).item()

        prob_real = (
            1.0 - prob_fake
        )

    print("\n====================")
    print(
        f"Raw Logit: "
        f"{raw_logit:.4f}"
    )

    print(
        f"Fake Probability: "
        f"{prob_fake:.6f}"
    )

    print(
        f"Real Probability: "
        f"{prob_real:.6f}"
    )
    print("====================")

    return {
        "Real Image": float(prob_real),
        "AI-Generated Image": float(prob_fake)
    }
demo = gr.Interface(
    fn=predict_uploaded_image,
    inputs=gr.Image(type="pil", label="Upload an Image to Analyze"),
    outputs=gr.Label(num_top_classes=2, label="Detection Probabilities"),
    title="AI-Generated Image Detector",
    description="Drop an image here to check if it's real or AI-generated using your active CvT-13 + CLIP model.",
    theme="soft"
)
demo.launch(share=True, debug=True)